# 메멘토(Memento) 패턴
객체의 상태를 저장해 두었다가, 필요할 때 객체를 해당 상태로 되돌릴 수 있게 해주는 행위 디자인 패턴. (like `Ctrl+Z`)
> '메멘토(Memento)'는 라틴어로 '기억하라' 또는 '기억(물)' 이라는 뜻


## 목적
- 시스템에서 핵심적인 기능을 담당하는 객체의 상태 저장
- 핵심적인 객체의 캡슐화 유지

## 장점
1. **캡슐화 유지** : 객체의 내부 상태를 외부에 노출하지 않고도 상태 보관이 가능
2. **복구의 용이성** : 복잡한 객체의 이전 상태를 관리자가 대신 관리해 주므로, 객체 자체가 과거 데이터를 직접 관리하는 부담을 덜어줌.
3. **단일 책임 원칙(SRP)**: 상태 저장 및 복구의 책임을 **Caretaker(관리자)** 에게 위임하여 역할을 분리할 수 있음.

## 주요 구성 요소
- Originator (작성자): 현재 상태를 가지고 있으며, **스냅샷을 찍거나 복구**하는 본체입니다.
- Memento (메멘토): 특정 시점의 상태를 담은 **수정 불가능한 기록물**입니다.
- Caretaker (관리자): 메멘토를 건드리지 않고 **순서대로 저장/전달**만 하는 관리자입니다.

In [1]:
import java.util.Stack

// 1. Memento: 상태를 저장하는 데이터 클래스 (불변성 유지)
data class GameStateMemento(
    val level: Int,
    val score: Int,
    val inventory: List<String>
)

// 2. Originator: 현재 게임의 상태를 관리하고 스냅샷을 생성/복구함
class GameCharacter {
    var level: Int = 1
    var score: Int = 0
    var inventory: MutableList<String> = mutableListOf()

    fun play(newScore: Int, newItem: String) {
        level++
        score += newScore
        inventory.add(newItem)
        println("진행 중... 현재 레벨: $level, 점수: $score, 인벤토리: $inventory")
    }

    // 현재 상태를 메멘토에 저장
    fun save(): GameStateMemento {
        println("--- 시스템: 현재 상태를 저장합니다 ---")
        // 리스트의 경우 참조만 복사되지 않도록 새로운 리스트로 생성
        return GameStateMemento(level, score, inventory.toList())
    }

    // 메멘토로부터 상태 복구
    fun restore(memento: GameStateMemento) {
        this.level = memento.level
        this.score = memento.score
        this.inventory = memento.inventory.toMutableList()
        println("--- 시스템: 이전 상태로 복구되었습니다 (레벨: $level) ---")
    }
}

// 3. Caretaker: 메멘토(세이브 파일)들을 관리
class GameSaveManager {
    private val savePoints = Stack<GameStateMemento>()

    fun save(memento: GameStateMemento) {
        savePoints.push(memento)
    }

    fun undo(): GameStateMemento? {
        return if (savePoints.isNotEmpty()) savePoints.pop() else null
    }
}

val character = GameCharacter()
val saveManager = GameSaveManager()

// 게임 플레이 및 저장 1
character.play(100, "나무 검")
saveManager.save(character.save())

// 게임 플레이 및 저장 2
character.play(200, "철 갑옷")
saveManager.save(character.save())

// 보스전 실패 상황 가정
character.play(500, "전설의 검")
println("현재 상태: 레벨 ${character.level}, 인벤토리 ${character.inventory}")

// 복구 실행 (최근 저장 시점으로 이동)
val lastSave = saveManager.undo()
if (lastSave != null) {
    character.restore(lastSave)
}

println("복구 후 상태: 레벨 ${character.level}, 인벤토리 ${character.inventory}")

진행 중... 현재 레벨: 2, 점수: 100, 인벤토리: [나무 검]
--- 시스템: 현재 상태를 저장합니다 ---
진행 중... 현재 레벨: 3, 점수: 300, 인벤토리: [나무 검, 철 갑옷]
--- 시스템: 현재 상태를 저장합니다 ---
진행 중... 현재 레벨: 4, 점수: 800, 인벤토리: [나무 검, 철 갑옷, 전설의 검]
현재 상태: 레벨 4, 인벤토리 [나무 검, 철 갑옷, 전설의 검]
--- 시스템: 이전 상태로 복구되었습니다 (레벨: 3) ---
복구 후 상태: 레벨 3, 인벤토리 [나무 검, 철 갑옷]


## 단점
1. **메모리 과부하** : 상태를 통째로 복사해서 저장하기 때문에, 자주 저장하거나 데이터가 크면 **메모리(RAM)** 를 엄청나게 잡아먹습니다.
2. **자원 관리의 어려움** : 저장된 기록(메멘토)이 쌓일수록 이를 관리하고 삭제하는 **보관소(Caretaker)** 의 로직이 복잡해집니다.
3. **복구 비용 발생** : 복잡한 객체를 매번 새로 생성해서 저장하고 복구하는 과정에서 CPU 자원을 많이 소모합니다.

> 이 단점들을 해결하기 위해 보통 **'최근 10개만 저장'** 하거나 **'바뀐 부분만 저장'** 하는 방식을 섞어서 씁니다.